   A quaternion is a compact way to represent a 3D ORIENTATION/ROTATION,
   usually written as (x, y, z, w). For a rotation by angle theta around a unit
   axis (ax, ay, az)

   ... singularity is nto automatically catastrophic: if the robot is moving 
   slowly, has free space, and can switch to joint-space motion, it can simply
   bend away. Singularities become dangerous when the controller must maintain
   a precise Carteisan trajectory, especially at speed or during contaxt. Near 
   one, you can get:
   - Enormous joint velocoities for a modest TCP velocity. 
   - Joint-velocity saturation, 

   the Jacobean matrix is a fundamental mathematical tool that maps joint-space
   velocities (like motor speeds or angles) to task-space velocities (the linear
   and angular velocity of the end-effector) via the equation
   x = J(q) q

---

   A singularity is not automatically catastrophic ... the keyword is NEAR: the
   exact singular configulation is only one point, but its neighbourhood can
   already e trobulesome because the required joint rates grow 
   approxiamtely like 1 / sigma-min. 

   ...

   For calculation, the tight answer is: 
   ... `A robot is kinematicall singular when its task-space Jacobian loses rank.`

   ---

   Ways to detect it:
   - General case: calculate `rank(J)` and check whether it is below the maximum
     possible rank.
   - Small 6x6 Jacobian: `det(J) = 0` indicates a singularity.
   - Numerically robust method: compute ...   

   In the context of Jacobian singularities, sigma-min refers to the smallest
   singular value of the robot's kinematic Jacobian matrix, J(q). Mathematically,
   applying Singular Value Decomposition (SVD) decomposes the Jacobian into 
   J = U \sum V^T, where the diagonal matrix \sum contains the singular values
   orderecd from largest sigma-max to smallest ... 

   When a robot approaches a kinemematic .... 

---

   Imagine you are trying to thread a needle, but a slight breeze keeps pushing
   your hand a millimeter to the left. If you only react to where your hand is
   at this exact second, you will constantly just correct that one
   millimeter, but the breeze will immediately push you back. You
   end up stuck, always slightly missing the hole. The variables 
   `tip_x_error_integrator` and `tip_y_error_integrator` act as the robot's 
   "memory" to solve this exact problem. Instead of just looking at the current
   mistake, the code adds up (integrates) all of the robot's pass misses. If it 
   realises it has been constantly off to the left for the last several seconds,
   these variables accumulate that error and tell the robot to push a little
   extra hard to the right, finally overcoming whatever invisible force (like
   a stiff cable or slight calibration error) was holding it back. 

   However, giving a robot memory can be dangerous if things go wrong. Suppose 
   the robot is trying to plug the cable in, but the plug gets physically
   snagged on the edge of the board. The robot ... stuck... becuase not moving
   ... those "integrator" variables will just keep adding up the error, growing
   larger and larger as the system gets more and more "frustrated." If the cable
   suddenly slipped free a few seconds later, that massive built-up memory
   woyuld command the robot to violently jerk the arm to overcompensate, 
   potentially breakign the hardware.

   This is why `max_integrator_windup` exists. It acts as a strict safety cap 
   (set to 0.0 meters in your code) on how big that error memory is allowed to
   get. 


```python
class CheatCode(Policy):
    def __init__(self, parent_node):
        self._tip_x_error_integrator = 0.0
        self._tip_y_error_integrator = 0.0
        self._max_integrator_windup = 0.05      # safety cap to prevent violent jerk due to over-correction, which could break the hardware.
        self._task = None
        super().__init__(parent_node)
```

---

   A `transform` describes the relationship between two frames. It contains:
   - A translation
   - A quaternion rotation
   - A timestamp
   - The names of the parent and child frames

   The TF buffer does not usually store every pose in every possible frame. It
   stores relationship between camera frames:

   world -> base_link -> wrist -> gripper/tcp

   world -> task_board -> port_link

   The bffer can combine these relationships when you make a request. For
   example: 

---
```python
lookup_transform("base_link", "port_link", Time())
```

   TF follows the chain between ... it combines the translations and rotations.
   It then returns the port pose in `base_link` coordinates. 




- `target_frame`: The frame you want to express the coordinates in (the 
  observer/reference frame)
- `source_frame`: The frame whose pose you want to know. 

   RESULT: It returns the pose of `source_frame` measured from the viewpoint of
   `target_frame`.

   

---

   ... 
When `groun-truth:=false`
   The buffer contains the normal robot TF tree. 
   - base_link
   - Robot arm links and joints
   - Wrist and tool links
   - gripper/tcp
   - Gripper links
   - Camera and sensor frames

   `robot_state_publisher` calculates these transforms from the robot 
   description and joint states. 


When `ground_truth:=true`
   The buffer contains the same normal robot transforms, but also contains
   simulator ground-truth transforms for:
   - the task board
   - components on the task board
   - port links
   - cable links
   - plug links
   - ohter published simulation objecct frames
   - the connnection between world and aic_world. 

(base) ➜  ~ distrobox enter -r aic_eval                              

(base) ➜  ~ /entrypoint.sh ground_truth:=false start_aic_engine:=true


(base) ➜  ~ pixi run ros2 run aic_model aic_model --ros-args -p use_sim_time:=true -p policy:=aic_example_policies.ros.WaveArm


---

   The primary objective of this code is to compute the target orientation for
   the TCP so that the plug it's holding perfectly aligns with the target port.
   Because the robot doesn't know exctly how the plug is seated within its
   grasp, it relies on live transform (TF) lookups ... 
      `self._parent_node._tf_buffer._lookup_transform`
   lookups rather than static assumptions. It first extracts the orientation
   quaternion of the target (`q_port`) and queries the active 

   - a standard SLERP function does not return a full list of waypoints. It 
     returns a single interpolated rotation (as a single quaternion) for a
     specific, single input time or ratio value between 0.0 and 1.0.

   - Take two orientations (start and end quaternions)
   - Takes one scalar value (the interpolation factor t, [0, 1])
   - Returns one single intermediate orientation

   







1. The Port Pose in the Base Frame (The absolute map)
   Before the robot can do any math, it needs a common language. The gripper,
   the camera, and the task board might all have their own localised coordinates
   , but the `base_link` (the physical bottom of the robot arm) acts as the
   universal map for the entire system. 

   When the code looks up the port pose in the base frame (`port_xy`), it is
   finding the absolute "bullseey" coordinates for the whole...  Once both the
   robot's hand and the port are plotted on this exact same 3D map, calculating
   the distance between them becomes simple subtraction. 


2. CALCULATING X/Y TARGETS && `i_gain`
   Once the code knows the exact bullseye (`port_xy`), it calculates where to 
   move using this math ...
      `target_x = port_transform.translation_x + i_gain * self._tip_x_error_integrator`
      Your ... `i_gain` (Integral Gain) is a scalar multiplier. It acts as a 
      volume knob for the error memory. 
   - THE MATH: If the robo ... 
   - THE RESULT: By multiplying ... only applies 15% of stored error per calculation step


3. WHY `z_offset` CHANGES (HOVER vs. INSERT)
   The `z_offset` is a vertical standoff ditance, and changing it in two distinct
   phases is a fundamental safety practice in robotics.
